# YOLO11 豬隻檢測器訓練

這個筆記本準備數據、訓練並評估用於豬隻檢測的YOLO11檢測器。我們直接從YAML架構（`yolo11m.yaml`）實例化網路並使用隨機權重，利用RTX 4090友好的超參數，並輸出Kaggle準備的提交檔案。

In [1]:
# 環境設置
from pathlib import Path
import random
import shutil
import yaml

import numpy as np
import pandas as pd
from PIL import Image

import torch

try:
    from ultralytics import YOLO
    from ultralytics.utils import ROOT as ULTRA_ROOT
except ImportError as exc:
    raise ImportError(
        "需要Ultralytics套件。請使用 `pip install -U ultralytics` 安裝以使用YOLOv11模型。"
    ) from exc



Torch版本: 2.4.1+cu121
使用設備: cuda


In [4]:
# 資料夾路徑以及超參數定義

DATA_ROOT = Path(".").resolve()
TRAIN_IMG_DIR = DATA_ROOT / "train" / "img"
TRAIN_GT_PATH = DATA_ROOT / "train" / "gt.txt"
TEST_IMG_DIR = DATA_ROOT / "test" / "img"
SAMPLE_SUB_PATH = DATA_ROOT / "sample_submission.csv"
YOLO_OUTPUT_DIR = DATA_ROOT / "yolo_runs"
YOLO_DATA_ROOT = DATA_ROOT / "yolo_dataset"
SUBMISSION_PATH = DATA_ROOT / "submission_yolo.csv"

CFG_NAME = "yolo11.yaml"
MODEL_CFG_ROOT = ULTRA_ROOT / "cfg" / "models"

candidates = [p for p in MODEL_CFG_ROOT.rglob(CFG_NAME) if p.is_file()]
if not candidates:
    local_candidate = DATA_ROOT / CFG_NAME
    if local_candidate.exists():
        candidates.append(local_candidate.resolve())

if not candidates:
    raise FileNotFoundError(
        f"在{MODEL_CFG_ROOT}或目前工作區內找不到所需的架構檔案{CFG_NAME}。"
        "請確認`pip install -U ultralytics`已成功完成，或將YAML檔案放置在此筆記本旁邊。"
    )

YOLO_CFG_PATH = candidates[0].resolve()
print(f"使用YOLO配置: {YOLO_CFG_PATH}")

VAL_SPLIT = 0.2
SEED = 13324837

YOLO_EPOCHS = 50  # 從頭開始收斂需要更長的訓練週期
YOLO_BATCH = 10 # 更高的批次大小適合原生解析度
YOLO_IMAGE_SIZE = 1024  # 匹配原生寬度/高度倍數 (640x360)
YOLO_LR0 = 0.005
YOLO_WEIGHT_DECAY = 3e-3
YOLO_MOMENTUM = 0.937
YOLO_WARMUP_EPOCHS = 8.0
YOLO_PATIENCE = 35

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"訓練圖片: {len(list(TRAIN_IMG_DIR.glob('*.jpg')))}")
print(f"測試圖片: {len(list(TEST_IMG_DIR.glob('*.jpg')))}")

使用YOLO配置: /home/tony/anaconda3/envs/CV/lib/python3.11/site-packages/ultralytics/cfg/models/11/yolo11.yaml
訓練圖片: 1266
測試圖片: 1864


In [5]:
# 將真實標註轉換為YOLO格式並建立資料集配置
YOLO_IMAGES_DIR = YOLO_DATA_ROOT / "images"
YOLO_LABELS_DIR = YOLO_DATA_ROOT / "labels"
for split in ["train", "val", "test"]:
    (YOLO_IMAGES_DIR / split).mkdir(parents=True, exist_ok=True)
    (YOLO_LABELS_DIR / split).mkdir(parents=True, exist_ok=True)

annotations_df = pd.read_csv(TRAIN_GT_PATH, header=None, names=["frame", "x", "y", "w", "h"])
annotations_map = {}
for _, row in annotations_df.iterrows():
    annotations_map.setdefault(int(row["frame"]), []).append((float(row["x"]), float(row["y"]), float(row["w"]), float(row["h"])) )

all_images = sorted(TRAIN_IMG_DIR.glob("*.jpg"))
indices = torch.randperm(len(all_images)).tolist()
val_count = max(1, int(len(all_images) * VAL_SPLIT))

split_assignments = {}
for position, original_idx in enumerate(indices):
    img_path = all_images[original_idx]
    # 根據在打亂列表中的位置決定split
    split_assignments[img_path] = "val" if position < val_count else "train"

for img_path in sorted(TEST_IMG_DIR.glob("*.jpg")):
    split_assignments[img_path] = "test"

invalid_box_counter = 0
for img_path, split in split_assignments.items():
    target_img_dir = YOLO_IMAGES_DIR / split
    target_lbl_dir = YOLO_LABELS_DIR / split

    target_img_path = target_img_dir / img_path.name
    if not target_img_path.exists():
        try:
            target_img_path.symlink_to(img_path)
        except FileExistsError:
            pass
        except OSError:
            shutil.copy2(img_path, target_img_path)

    if split in {"train", "val"}:
        image = Image.open(img_path).convert("RGB")
        width, height = image.size
        boxes = annotations_map.get(int(img_path.stem), [])

        yolo_lines = []
        for x, y, w, h in boxes:
            if w <= 0 or h <= 0:
                invalid_box_counter += 1
                continue
            x_center = (x + w / 2.0) / width
            y_center = (y + h / 2.0) / height
            w_norm = w / width
            h_norm = h / height

            if not (0.0 < w_norm <= 1.0 and 0.0 < h_norm <= 1.0):
                invalid_box_counter += 1
                continue

            x_center = min(max(x_center, 0.0), 1.0)
            y_center = min(max(y_center, 0.0), 1.0)
            w_norm = min(max(w_norm, 1e-6), 1.0)
            h_norm = min(max(h_norm, 1e-6), 1.0)

            yolo_lines.append(f"0 {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}")

        label_path = target_lbl_dir / f"{img_path.stem}.txt"
        with label_path.open("w", encoding="utf-8") as f:
            f.write("\n".join(yolo_lines))
    else:
        # 為測試圖片建立空的標籤檔案（可選但保持結構一致）
        label_path = target_lbl_dir / f"{img_path.stem}.txt"
        if not label_path.exists():
            label_path.touch()

if invalid_box_counter:
    print(f"轉換過程中跳過了{invalid_box_counter}個無效的邊界框")

train_count = len(list((YOLO_IMAGES_DIR / "train").glob("*.jpg")))
val_count = len(list((YOLO_IMAGES_DIR / "val").glob("*.jpg")))
test_count = len(list((YOLO_IMAGES_DIR / "test").glob("*.jpg")))
print(f"YOLO資料集 -> 訓練:{train_count} | 驗證:{val_count} | 測試:{test_count}")

yolo_data_yaml = YOLO_DATA_ROOT / "pig_detection.yaml"
yolo_data_dict = {
    "path": str(YOLO_DATA_ROOT.resolve()),
    "train": str((YOLO_IMAGES_DIR / "train").resolve()),
    "val": str((YOLO_IMAGES_DIR / "val").resolve()),
    "test": str((YOLO_IMAGES_DIR / "test").resolve()),
    "nc": 1,
    "names": ["pig"],
}
with yolo_data_yaml.open("w", encoding="utf-8") as f:
    yaml.safe_dump(yolo_data_dict, f)

轉換過程中跳過了3個無效的邊界框
YOLO資料集 -> 訓練:1013 | 驗證:253 | 測試:1864
已寫入資料集配置 -> /home/tony/Downloads/NTU_CVPDL/H1/src/yolo_dataset/pig_detection.yaml


In [6]:
# 從YAML定義訓練YOLO（隨機初始化）
yolo_device = 0 if torch.cuda.is_available() else "cpu"

model = YOLO('yolo11m.yaml')
train_results = model.train(
    data=str(yolo_data_yaml),
    epochs=YOLO_EPOCHS,
    imgsz=YOLO_IMAGE_SIZE,
    batch=YOLO_BATCH,
    device=yolo_device,
    pretrained=False,
    lr0=YOLO_LR0,
    momentum=YOLO_MOMENTUM,
    weight_decay=YOLO_WEIGHT_DECAY,
    warmup_epochs=YOLO_WARMUP_EPOCHS,
    patience=YOLO_PATIENCE,
    cache=True,
    workers=8,
    cos_lr=True,
    amp=True,
    # rect=True,
    # multi_scale=True,
    close_mosaic=10,
    dropout=0.3,
    mixup=0.2,
    label_smoothing=0.03,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    translate=0.08,
    scale=0.6,
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    save_period=50,
    project=str(YOLO_OUTPUT_DIR),
    name="yolo11x_pig_from_yaml2",
    exist_ok=True,
)

trainer = train_results if hasattr(train_results, "save_dir") else getattr(model, "trainer", None)
if trainer is None or not hasattr(trainer, "save_dir"):
    raise RuntimeError("無法找到YOLO訓練器的save_dir。請檢查ultralytics版本和訓練運行。")

best_weights = Path(trainer.save_dir) / "weights" / "best.pt"
last_weights = Path(trainer.save_dir) / "weights" / "last.pt"
if best_weights.exists():
    print(f"載入最佳檢查點: {best_weights}")
    best_model = YOLO(best_weights)
elif last_weights.exists():
    print("未找到最佳權重；回退到最後一個週期的權重。")
    best_model = YOLO(last_weights)
else:
    print("未找到已儲存的權重；使用記憶體中的模型實例。")
    best_model = model

在設備0上開始訓練
New https://pypi.org/project/ultralytics/8.3.214 available 😃 Update with 'pip install -U ultralytics'
WARNING ⚠️ 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.3.207 🚀 Python-3.11.10 torch-2.4.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4090, 24564MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=10, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/home/tony/Downloads/NTU_CVPDL/H1/src/yolo_dataset/pig_detection.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.3, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.01, mask_ratio=4, ma

KeyboardInterrupt: 

In [13]:
# 在驗證集上評估以獲得快速回饋
val_metrics = best_model.val(
    data=str(yolo_data_yaml),
    imgsz=YOLO_IMAGE_SIZE,
    batch=YOLO_BATCH,
    device=yolo_device,
    split="val",
    save_json=False,
    verbose=True,
)
print(val_metrics)

Ultralytics 8.3.207 🚀 Python-3.11.10 torch-2.4.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4090, 24564MiB)
YOLO11m summary (fused): 125 layers, 20,030,803 parameters, 0 gradients, 67.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 985.5±853.6 MB/s, size: 22.1 KB)
val: Scanning /home/tony/Downloads/NTU_CVPDL/H1/yolo_dataset/labels/val.cache... 253 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 253/253 610.6Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 26/26 8.9it/s 2.9s<0.2s
                   all        253       7677      0.984      0.975      0.992      0.865
Speed: 0.4ms preprocess, 6.7ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to /home/tony/Downloads/NTU_CVPDL/H1/runs/detect/val195
ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix objec

In [14]:

# 明確打印不同IoU閾值的mAP
print("=== 詳細mAP指標 ===")
print(f"mAP50: {val_metrics.box.map50:.4f}")      # IoU=0.5時的mAP
print(f"mAP50-95: {val_metrics.box.map:.4f}")     # IoU=0.5-0.95的平均mAP
print(f"mAP75: {val_metrics.box.map75:.4f}")      # IoU=0.75時的mAP

# 如果有各類別的mAP
if hasattr(val_metrics.box, 'maps'):
    print(f"Per-class mAP50-95: {val_metrics.box.maps}")

=== 詳細mAP指標 ===
mAP50: 0.9923
mAP50-95: 0.8646
mAP75: 0.9330
Per-class mAP50-95: [    0.86457]


In [15]:
# 在驗證集上掃描置信度/IoU閾值以選擇最佳提交操作點
import numpy as np

conf_grid = np.linspace(0.10, 0.40, 7)
iou_grid = [0.4, 0.5, 0.55, 0.6, 0.65]  # 為NMS調優調整IoU閾值
records = []
best_model = model
yolo_device = 0 if torch.cuda.is_available() else "cpu"

for conf in conf_grid:
    for iou in iou_grid:
        metrics = best_model.val(
            data=str(yolo_data_yaml),
            imgsz=YOLO_IMAGE_SIZE,
            batch=YOLO_BATCH,
            device=yolo_device,
            split="val",
            conf=conf,
            iou=iou,
            verbose=False,
        )
        records.append({
            "conf": conf,
            "iou": iou,
            "map50": metrics.box.map50,
            "map": metrics.box.map,
        })

import pandas as pd
threshold_df = pd.DataFrame(records).sort_values(by=["map", "map50"], ascending=False)
threshold_df.reset_index(drop=True, inplace=True)
threshold_df.head()

BEST_CONF_THRESHOLD = float(threshold_df.iloc[0]["conf"]) if not threshold_df.empty else 0.25
BEST_IOU_THRESHOLD = float(threshold_df.iloc[0]["iou"]) if not threshold_df.empty else 0.6
print(f"選定的閾值 -> 置信度: {BEST_CONF_THRESHOLD:.3f}, IoU: {BEST_IOU_THRESHOLD:.2f}")

Ultralytics 8.3.207 🚀 Python-3.11.10 torch-2.4.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4090, 24564MiB)
YOLO11m summary (fused): 125 layers, 20,030,803 parameters, 0 gradients, 67.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2221.4±796.9 MB/s, size: 22.4 KB)
val: Scanning /home/tony/Downloads/NTU_CVPDL/H1/yolo_dataset/labels/val.cache... 253 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 253/253 657.9Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 26/26 9.9it/s 2.6s<0.2s
                   all        253       7677      0.983      0.947      0.974       0.87
Speed: 0.5ms preprocess, 5.3ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to /home/tony/Downloads/NTU_CVPDL/H1/runs/detect/val196
Ultralytics 8.3.207 🚀 Python-3.11.10 torch-2.4.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4090, 24564MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2538.1±991.9 MB/s, size: 23.5 KB)
val: Scanning /ho

In [35]:
BEST_CONF_THRESHOLD = 0.001  # 根據驗證集結果微調
BEST_IOU_THRESHOLD = 0.68

In [36]:
from tqdm import tqdm

# 使用訓練好的YOLO模型和輕量TTA生成提交CSV
@torch.no_grad()
def create_submission(model, sample_path, test_dir, output_path, conf_threshold=BEST_CONF_THRESHOLD, iou_threshold=BEST_IOU_THRESHOLD):
    sample_df = pd.read_csv(sample_path)
    predictions = []

    # 加入tqdm進度條
    for image_id in tqdm(sample_df["Image_ID"], desc="生成預測結果", unit="張"):
        img_path = Path(test_dir) / f"{int(image_id):08d}.jpg"
        if not img_path.exists():
            predictions.append("")
            continue

        results = model.predict(
            source=str(img_path),
            imgsz=YOLO_IMAGE_SIZE,
            conf=conf_threshold,
            iou=iou_threshold,
            device=yolo_device,
            augment=True,
            verbose=False,
        )

        if not results:
            predictions.append("")
            continue

        boxes = results[0].boxes
        if boxes is None or boxes.xyxy is None or boxes.xyxy.numel() == 0:
            predictions.append("")
            continue

        xyxy = boxes.xyxy.cpu().numpy()
        scores = boxes.conf.cpu().numpy()
        classes = boxes.cls.cpu().numpy()

        entries = []
        for score, box, cls in zip(scores, xyxy, classes):
            x_min, y_min, x_max, y_max = box
            width = max(0.0, x_max - x_min)
            height = max(0.0, y_max - y_min)
            entries.append(f"{score:.4f} {x_min:.1f} {y_min:.1f} {width:.1f} {height:.1f} {int(cls)}")

        predictions.append(" ".join(entries))

    submission_df = sample_df.copy()
    submission_df["PredictionString"] = predictions
    submission_df.to_csv(output_path, index=False)
    return submission_df

submission_df = create_submission(best_model, SAMPLE_SUB_PATH, TEST_IMG_DIR, SUBMISSION_PATH)
# submission_df = create_submission(best_model, SAMPLE_SUB_PATH, TEST_IMG_DIR, SUBMISSION_PATH, conf_threshold=0.15, iou_threshold=0.65)

print(f"提交檔案已儲存至 {SUBMISSION_PATH}")
submission_df.head()

生成預測結果:   0%|          | 0/1864 [00:00<?, ?張/s]

生成預測結果: 100%|██████████| 1864/1864 [00:50<00:00, 36.73張/s]


提交檔案已儲存至 /home/tony/Downloads/NTU_CVPDL/H1/submission_yolo.csv


,Image_ID,PredictionString
0,1,0.8906 84.3 288.2 162.5 71.8 0 0.8793 181.0 63...
1,2,0.8890 87.5 286.2 161.3 73.8 0 0.8712 182.0 64...
2,3,0.8957 86.7 287.4 165.1 72.6 0 0.8815 184.7 65...
3,4,0.9013 85.7 289.9 161.9 69.6 0 0.8739 184.6 65...
4,5,0.8828 186.0 65.6 117.6 72.1 0 0.8558 71.8 107...


In [37]:
# 詳細分析提交結果的統計資訊
def analyze_submission_results(submission_df):
    """分析提交結果的詳細統計資訊"""
    print("=" * 60)
    print("📊 提交結果統計分析")
    print("=" * 60)
    
    total_images = len(submission_df)
    
    # 統計每張圖片的檢測資訊
    detection_stats = []
    bbox_counts = []
    confidence_scores = []
    bbox_sizes = []
    
    for idx, row in submission_df.iterrows():
        pred_string = row['PredictionString']
        if pd.isna(pred_string) or pred_string == '':
            bbox_counts.append(0)
            detection_stats.append({
                'image_id': row['Image_ID'],
                'bbox_count': 0,
                'max_conf': 0,
                'avg_conf': 0,
                'avg_bbox_area': 0
            })
        else:
            # 解析預測字串：score x_min y_min width height class_id
            predictions = pred_string.strip().split(' ')
            num_predictions = len(predictions) // 6  # 每6個元素組成一個預測
            bbox_counts.append(num_predictions)
            
            image_confidences = []
            image_areas = []
            
            for i in range(num_predictions):
                try:
                    score = float(predictions[i * 6])
                    width = float(predictions[i * 6 + 3])
                    height = float(predictions[i * 6 + 4])
                    
                    confidence_scores.append(score)
                    image_confidences.append(score)
                    
                    area = width * height
                    bbox_sizes.append(area)
                    image_areas.append(area)
                except (ValueError, IndexError):
                    continue
            
            detection_stats.append({
                'image_id': row['Image_ID'],
                'bbox_count': num_predictions,
                'max_conf': max(image_confidences) if image_confidences else 0,
                'avg_conf': np.mean(image_confidences) if image_confidences else 0,
                'avg_bbox_area': np.mean(image_areas) if image_areas else 0
            })
    
    # 基本統計
    images_with_detections = sum(1 for count in bbox_counts if count > 0)
    images_without_detections = total_images - images_with_detections
    total_bboxes = sum(bbox_counts)
    
    print(f"🎯 檢測概覽：")
    print(f"   總圖片數量: {total_images:,}")
    print(f"   有檢測結果: {images_with_detections:,} ({images_with_detections/total_images*100:.1f}%)")
    print(f"   無檢測結果: {images_without_detections:,} ({images_without_detections/total_images*100:.1f}%)")
    print(f"   總檢測框數: {total_bboxes:,}")
    
    # Bbox數量統計
    print(f"\n📦 邊界框統計：")
    print(f"   平均每張圖片bbox數: {np.mean(bbox_counts):.2f}")
    print(f"   bbox數量中位數: {np.median(bbox_counts):.1f}")
    print(f"   最大bbox數(單張圖): {max(bbox_counts) if bbox_counts else 0}")
    print(f"   bbox數量標準差: {np.std(bbox_counts):.2f}")
    
    # Bbox數量分布
    if bbox_counts:
        bbox_distribution = {}
        for count in bbox_counts:
            bbox_distribution[count] = bbox_distribution.get(count, 0) + 1
        
        print(f"\n📈 Bbox數量分布（前10名）：")
        sorted_dist = sorted(bbox_distribution.items(), key=lambda x: x[1], reverse=True)[:10]
        for bbox_num, freq in sorted_dist:
            percentage = freq / total_images * 100
            print(f"   {bbox_num:2d}個bbox: {freq:4d}張圖片 ({percentage:5.1f}%)")
    
    # 置信度統計
    if confidence_scores:
        print(f"\n🎯 置信度統計：")
        print(f"   平均置信度: {np.mean(confidence_scores):.4f}")
        print(f"   置信度中位數: {np.median(confidence_scores):.4f}")
        print(f"   最高置信度: {max(confidence_scores):.4f}")
        print(f"   最低置信度: {min(confidence_scores):.4f}")
        print(f"   置信度標準差: {np.std(confidence_scores):.4f}")
        
        # 置信度區間分布
        conf_ranges = {
            '0.9-1.0': sum(1 for c in confidence_scores if 0.9 <= c <= 1.0),
            '0.7-0.9': sum(1 for c in confidence_scores if 0.7 <= c < 0.9),
            '0.5-0.7': sum(1 for c in confidence_scores if 0.5 <= c < 0.7),
            '0.3-0.5': sum(1 for c in confidence_scores if 0.3 <= c < 0.5),
            '0.1-0.3': sum(1 for c in confidence_scores if 0.1 <= c < 0.3),
            '0.0-0.1': sum(1 for c in confidence_scores if 0.0 <= c < 0.1),
        }
        
        print(f"\n📊 置信度區間分布：")
        for range_name, count in conf_ranges.items():
            percentage = count / len(confidence_scores) * 100
            print(f"   {range_name}: {count:4d} ({percentage:5.1f}%)")
    
    # 邊界框大小統計
    if bbox_sizes:
        print(f"\n📏 邊界框尺寸統計：")
        print(f"   平均面積: {np.mean(bbox_sizes):.1f} 像素²")
        print(f"   面積中位數: {np.median(bbox_sizes):.1f} 像素²")
        print(f"   最大面積: {max(bbox_sizes):.1f} 像素²")
        print(f"   最小面積: {min(bbox_sizes):.1f} 像素²")
        
        # 尺寸分布
        small_boxes = sum(1 for size in bbox_sizes if size < 1000)
        medium_boxes = sum(1 for size in bbox_sizes if 1000 <= size < 10000)
        large_boxes = sum(1 for size in bbox_sizes if size >= 10000)
        
        print(f"\n📐 尺寸分布：")
        print(f"   小框 (<1000px²): {small_boxes} ({small_boxes/len(bbox_sizes)*100:.1f}%)")
        print(f"   中框 (1000-10000px²): {medium_boxes} ({medium_boxes/len(bbox_sizes)*100:.1f}%)")
        print(f"   大框 (≥10000px²): {large_boxes} ({large_boxes/len(bbox_sizes)*100:.1f}%)")
    
    # 高檢測數量的圖片
    high_detection_images = [(stats['image_id'], stats['bbox_count']) 
                           for stats in detection_stats if stats['bbox_count'] > 10]
    if high_detection_images:
        print(f"\n🔥 高檢測數量圖片 (>10個bbox)：")
        high_detection_images.sort(key=lambda x: x[1], reverse=True)
        for img_id, count in high_detection_images[:5]:  # 顯示前5張
            print(f"   圖片 {img_id:08d}: {count} 個bbox")
    
    # 高置信度檢測
    if confidence_scores:
        high_conf_detections = sum(1 for c in confidence_scores if c > 0.8)
        print(f"\n⭐ 高置信度檢測 (>0.8): {high_conf_detections} 個 ({high_conf_detections/len(confidence_scores)*100:.1f}%)")
    
    print("=" * 60)
    
    return {
        'total_images': total_images,
        'images_with_detections': images_with_detections,
        'total_bboxes': total_bboxes,
        'avg_bboxes_per_image': np.mean(bbox_counts),
        'avg_confidence': np.mean(confidence_scores) if confidence_scores else 0,
        'bbox_counts': bbox_counts,
        'confidence_scores': confidence_scores,
        'detection_stats': detection_stats
    }
    
analyze_submission_results(submission_df)

# 在你的預測完成後調用這個函數
# print(f"使用 {custom_weight_path} 的預測已儲存至 {custom_submission_path}")

# 分析結果
# stats = analyze_submission_results(submission_df)

# 顯示DataFrame前幾行
# custom_submission_df.head()

📊 提交結果統計分析
🎯 檢測概覽：
   總圖片數量: 1,864
   有檢測結果: 1,864 (100.0%)
   無檢測結果: 0 (0.0%)
   總檢測框數: 448,938

📦 邊界框統計：
   平均每張圖片bbox數: 240.85
   bbox數量中位數: 243.0
   最大bbox數(單張圖): 300
   bbox數量標準差: 61.61

📈 Bbox數量分布（前10名）：
   300個bbox:  902張圖片 ( 48.4%)
   170個bbox:   17張圖片 (  0.9%)
   180個bbox:   17張圖片 (  0.9%)
   169個bbox:   16張圖片 (  0.9%)
   168個bbox:   16張圖片 (  0.9%)
   181個bbox:   16張圖片 (  0.9%)
   199個bbox:   16張圖片 (  0.9%)
   194個bbox:   15張圖片 (  0.8%)
   193個bbox:   15張圖片 (  0.8%)
   176個bbox:   14張圖片 (  0.8%)

🎯 置信度統計：
   平均置信度: 0.1208
   置信度中位數: 0.0111
   最高置信度: 0.9641
   最低置信度: 0.0010
   置信度標準差: 0.2395

📊 置信度區間分布：
   0.9-1.0: 1669 (  0.4%)
   0.7-0.9: 32159 (  7.2%)
   0.5-0.7: 16087 (  3.6%)
   0.3-0.5: 14695 (  3.3%)
   0.1-0.3: 32232 (  7.2%)
   0.0-0.1: 352096 ( 78.4%)

📏 邊界框尺寸統計：
   平均面積: 4164.2 像素²
   面積中位數: 2651.9 像素²
   最大面積: 48500.5 像素²
   最小面積: 0.0 像素²

📐 尺寸分布：
   小框 (<1000px²): 54763 (12.2%)
   中框 (1000-10000px²): 353068 (78.6%)
   大框 (≥10000px²): 41107 (9.2%)

🔥 高檢測數量圖片 (>10個b

{'total_images': 1864,
 'images_with_detections': 1864,
 'total_bboxes': 448938,
 'avg_bboxes_per_image': np.float64(240.84656652360516),
 'avg_confidence': np.float64(0.1207749145761776),
 'bbox_counts': [300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300,
  300